# Synthetic Validation Results

Presentation notebook for studies A–E.  
**Requires:** ANSYS MAPDL + pyFBS installed and `truth.json` / `parent.json` in
`synthetic_validation/configs/`.  
All logic lives in `synthetic_validation/studies.py`; this notebook only calls + plots.

| Study | Question |
|-------|----------|
| A     | Can the method recover the truth stress field (realistic vs oracle)? |
| B     | How does camera noise affect the recovered PSD distribution? |
| C     | How does FE-model discrepancy degrade accuracy? |
| D     | How many modes are needed for convergence? |
| E     | What is the conditioning of the mode-shape matrix? |

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri

from synthetic_validation.config import load_config
from synthetic_validation.studies import (
    study_a_recovery,
    study_b_noise,
    study_c_fe_discrepancy,
    study_d_modal_convergence,
    study_e_conditioning,
)

CONFIGS = Path("../configs")
truth_cfg  = load_config(CONFIGS / "truth.json")
parent_cfg = load_config(CONFIGS / "parent.json")

# Common harness kwargs for all studies
KW = dict(saa_level=1.0, fs=2000, n_frames=40000, seed=42, n_modes=4)

## Study A — Realistic vs Oracle Recovery

In [ ]:
res_a = study_a_recovery(truth_cfg, parent_cfg, analytic=False, **KW)

for run_key, label in [("realistic", "Realistic (parent prior)"),
                        ("oracle",    "Oracle (truth prior)")]:
    m = res_a[run_key]["metrics"]
    print(f"{label}")
    for comp in ("SX", "SY", "SXY", "SX+SY"):
        print(f"  {comp:5s}  NRMSE={m['nrmse'][comp]:.4f}   MAC={m['mac'][comp]:.4f}")

In [ ]:
# Spatial maps: peak-frequency recovered vs truth for SX
freqs    = res_a["realistic"]["freqs"]
rec_sx   = res_a["realistic"]["recovered"]["SX"]
truth_sx = res_a["realistic"]["truth"]["SX"]

# Peak frequency index
fpk    = int(np.argmax(np.sum(np.abs(truth_sx), axis=1)))
nnodes = truth_sx.shape[1]

# Use imshow (1 x nnodes) -- node coordinates are not in the result dict;
# avoids the shape-fragile scatter(*truth_cfg.point_mass_xy[:2], ...).
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, data, title in zip(axes,
                            [rec_sx[fpk], truth_sx[fpk]],
                            [f"Recovered SX @ {freqs[fpk]:.1f} Hz",
                             f"Truth SX @ {freqs[fpk]:.1f} Hz"]):
    im = ax.imshow(data.reshape(1, nnodes), aspect="auto", cmap="RdBu_r")
    plt.colorbar(im, ax=ax)
    ax.set_xlabel("node index")
    ax.set_yticks([])
    ax.set_title(title)
plt.tight_layout()
plt.savefig("study_a_spatial_maps.png", dpi=150)
plt.show()

## Study B — Noise Confidence Bands

In [ ]:
SNR_LEVELS = [0.5, 1.0, 2.0, 5.0, 10.0]
res_b = study_b_noise(
    truth_cfg, parent_cfg,
    n_reps=10,
    snr_levels=SNR_LEVELS,
    analytic=False,
    n_segments=8,   # Welch averaging for realistic PSD variance
    **KW,
)

# Plot mean ± 1 std for SX at one representative SNR
comp = "SX"
snr_plot = SNR_LEVELS[-1]    # highest SNR (least noise)
band = res_b["confidence_bands"][comp][snr_plot]

# Average over nodes for a scalar PSD vs frequency view
mean_psd = band["mean"].mean(axis=1)   # (nfreq,)
std_psd  = band["std"].mean(axis=1)

# We need the frequency axis from a quick single run
freqs_b = freqs   # reuse from Study A if KW are identical

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(freqs_b, mean_psd, label=f"Mean (SNR={snr_plot})")
ax.fill_between(freqs_b,
                np.clip(mean_psd - std_psd, 1e-30, None),
                mean_psd + std_psd,
                alpha=0.3, label="±1σ")
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("SX PSD (node avg)")
ax.set_title(f"Study B — noise confidence bands (SNR={snr_plot}, n_reps={res_b['n_reps']})")
ax.legend()
plt.tight_layout()
plt.savefig("study_b_noise_bands.png", dpi=150)
plt.show()

## Study C — FE-Model Discrepancy

In [ ]:
import dataclasses

# Build parent variants: perturb E by ±5 %, ±10 %, ±20 %
perturbations = [-0.20, -0.10, -0.05, 0.0, 0.05, 0.10, 0.20]
variants = [dataclasses.replace(parent_cfg, E=parent_cfg.E * (1 + dp))
            for dp in perturbations]

res_c = study_c_fe_discrepancy(
    truth_cfg, variants,
    analytic=False,
    nominal_idx=3,   # dp=0.0 is at index 3 in perturbations
    **KW,
)

nrmse_sx = [v["metrics"]["nrmse"]["SX"] for v in res_c["variants"]]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(perturbations, nrmse_sx, "o-")
ax.axvline(0, ls="--", color="gray", label="nominal parent")
ax.set_xlabel("ΔE / E_parent")
ax.set_ylabel("NRMSE (SX)")
ax.set_title("Study C — NRMSE vs E-modulus discrepancy")
ax.legend()
plt.tight_layout()
plt.savefig("study_c_discrepancy.png", dpi=150)
plt.show()

print("Realistic − Oracle gap (SX):",
      res_c["realistic_oracle_gap"].get("SX", "n/a"))

## Study D — Modal Convergence

In [ ]:
MODE_COUNTS = [1, 2, 3, 4, 6, 8]

res_d = study_d_modal_convergence(
    truth_cfg, parent_cfg,
    mode_counts=MODE_COUNTS,
    analytic=False,
    **{k: v for k, v in KW.items() if k != "n_modes"},  # n_modes is swept
)

fig, ax = plt.subplots(figsize=(7, 4))
for comp in ("SX", "SY", "SXY"):
    ax.plot(res_d["mode_counts"], res_d["nrmse"][comp], "o-", label=comp)
ax.set_xlabel("Number of modes")
ax.set_ylabel("NRMSE")
ax.set_title("Study D — Modal convergence")
ax.legend()
plt.tight_layout()
plt.savefig("study_d_convergence.png", dpi=150)
plt.show()

## Study E — Conditioning

In [ ]:
res_e = study_e_conditioning(
    truth_cfg, parent_cfg,
    regularize=False,
    analytic=False,
    **KW,
)

print(f"Condition number of Ψ_cam : {res_e['condition_number']:.2f}")
print("NRMSE per component:")
for comp, val in res_e["metrics"]["nrmse"].items():
    print(f"  {comp:5s}  {val:.4f}")

# Also show how condition number varies with n_modes (from Study D)
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(res_d["mode_counts"], res_d["condition_numbers"], "s-")
ax.set_xlabel("Number of modes")
ax.set_ylabel("Condition number")
ax.set_title("Study E — Conditioning vs #modes")
plt.tight_layout()
plt.savefig("study_e_conditioning.png", dpi=150)
plt.show()